<a href="https://colab.research.google.com/github/YardenNahum/FinalProjectXAI/blob/main/Phase%20B%2026-1-R3/Code/AI%20Models/DiabitiesModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports and Initial Installtion Of Libraries**

In [ ]:
!pip install kagglehub[pandas-datasets]
!pip install category_encoders
!pip install lime
!pip install dice-ml
!pip install firebase requests beautifulsoup4 nltk --quiet
from firebase import firebase
import pandas as pd
import numpy as np
import warnings
import json
import os
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
import dice_ml

**Database Connection**

In [ ]:
DB_URL = 'https://xai-project-b65cc-default-rtdb.firebaseio.com/'
FBconn = firebase.FirebaseApplication(DB_URL, None)

Downloading the dataset from kaggle https://www.kaggle.com/datasets/mohankrishnathalla/diabetes-health-indicators-dataset?select=diabetes_dataset.csv


In [ ]:
# download dataset
path = kagglehub.dataset_download("mohankrishnathalla/diabetes-health-indicators-dataset")

#load dataset to path
file_name = "diabetes_dataset.csv"
full_path = os.path.join(path, file_name)

df = pd.read_csv(full_path)

**Preprocessing The Data**

In [ ]:
# Seperate numirical and categorial colums
# Numerical columns
num_cols = [
    "age", "alcohol_consumption_per_week", "physical_activity_minutes_per_week",
    "diet_score", "sleep_hours_per_day", "screen_time_hours_per_day", "bmi",
    "waist_to_hip_ratio", "systolic_bp", "diastolic_bp", "heart_rate",
    "cholesterol_total", "hdl_cholesterol", "ldl_cholesterol", "triglycerides"
]

# Categorical columns
cat_cols = [
    "gender", "ethnicity", "education_level", "income_level", "smoking_status",
    "employment_status", "family_history_diabetes", "hypertension_history",
    "cardiovascular_history"
]

#Preprocess the data colums
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)





**Data Preparation and Splitting**

In [ ]:
# Separate Features and Target
target = "diagnosed_diabetes"

# Drop non-feature columns
X = df.drop(columns=[target, "id", "dataset", "diabetes_stage"], errors="ignore")
Y = df[target]

# K-Fold & Diagnostics
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

**Training and Testing**

In [ ]:

print("Training the model")
fold_accuracies = []

# K-Fold evaluation
for fold, (train_idx, test_idx) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]

    fold_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, random_state=42))
    ])

    fold_model.fit(X_train, Y_train)

    predictions = fold_model.predict(X_test)
    score = accuracy_score(Y_test, predictions)

    fold_accuracies.append(score)
    print(f"Fold {fold+1} Accuracy: {score:.4f}")

print(f"\nAverage Project Accuracy: {np.mean(fold_accuracies):.4f}")

# Final model for explanations and real use
full_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, random_state=42))
])

full_model.fit(X, Y)

In [ ]:
def Ensure_Directory_Exists(folder_path):
    os.makedirs(folder_path, exist_ok=True)

**Creating The SHAP LIME and DiCE**



In [ ]:
def Get_Lime_Explantion(patient_processed, X_processed, feature_names, model, record_id, output_dir="plots/diabetes"):
    Ensure_Directory_Exists(output_dir)

    explainer = LimeTabularExplainer(
        training_data=X_processed,
        feature_names=feature_names,
        class_names=['Healthy', 'Diabetes'],
        mode='classification'
    )

    exp = explainer.explain_instance(
        patient_processed[0],
        model.predict_proba
    )

    lime_features = [
        {"feature": feature, "weight": round(float(weight), 4)}
        for feature, weight in exp.as_list()
    ]
    local_probs = exp.predict_proba
    lime_html_path = os.path.join(output_dir, f"lime_record_{record_id}.html")
    exp.save_to_file(lime_html_path)
    exp.show_in_notebook(show_table=True)
    return {
        "Prediction":{
          "Healthy": round(float(local_probs[0]), 4),
          "Diabetes": round(float(local_probs[1]), 4)},
        "features": lime_features,
        "Type": "Tabular",
        "html_path": lime_html_path
    }

In [ ]:
def Get_Shap_Explantion(patient_row, X_processed, features_names, model, record_id, output_dir="plots/diabetes"):
    Ensure_Directory_Exists(output_dir)

    background = shap.sample(X_processed, min(100, len(X_processed)))
    prediction = int(model.predict(patient_row)[0])

    explainer = shap.KernelExplainer(model.predict_proba, background)
    shap_vals = explainer.shap_values(patient_row)

    base_val = float(explainer.expected_value[prediction])

    if isinstance(shap_vals, list):
        weights = shap_vals[prediction][0]
    else:
        weights = shap_vals[0, :, prediction]

    exp = shap.Explanation(
        values=weights,
        base_values=base_val,
        data=patient_row[0],
        feature_names=features_names
    )

    shap_png_path = os.path.join(output_dir, f"shap_record_{record_id}.png")

    plt.figure()
    shap.plots.bar(exp, show=True)
    plt.savefig(shap_png_path, bbox_inches="tight")
    plt.close()

    shap_json = {
        "base_value": round(base_val, 4),
        "prediction_probability": round(float(model.predict_proba(patient_row)[0][prediction]), 4),
        "feature_impacts": [
            {"feature": col, "weight": round(float(w), 4)}
            for col, w in zip(features_names, weights)
        ],
        "png_path": shap_png_path
    }

    return shap_json

In [ ]:
import pandas as pd
import warnings
import dice_ml

def Get_Dice_Explantion(query_instance, dataset_X, dataset_y, trained_model, numeric_features, categorical_features):
    warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

    # Combine features and target into one DataFrame for DiCE
    dice_dataframe = dataset_X.copy()
    dice_dataframe['target'] = dataset_y.values

    # Features that should not be changed
    fixed_features = ["gender", "age", "ethnicity", "education_level", "income_level", "employment_status"]

    #Get current prediction AND probability
    current_prediction = trained_model.predict(query_instance)[0]
    probabilities = trained_model.predict_proba(query_instance)[0]

    target_class = 0 if current_prediction == 1 else 1
    target_label = "Healthy" if target_class == 0 else "Diabetic"

    # Save the starting probability for our target class
    current_target_prob = probabilities[target_class]

    # Allow only non-fixed features to vary
    features_allowed_to_change = [
        feature for feature in dataset_X.columns
        if feature not in fixed_features
    ]

    # Initialize DiCE data interface with raw features
    dice_data = dice_ml.Data(
        dataframe=dice_dataframe,
        continuous_features=numeric_features,
        categorical_features=categorical_features,
        outcome_name='target'
    )

    # Wrap the trained pipeline model
    dice_model = dice_ml.Model(model=trained_model, backend="sklearn")

    # Initialize DiCE with genetic method
    dice_explainer = dice_ml.Dice(dice_data, dice_model, method="genetic")

    # Generate counterfactual examples
    dice_result = dice_explainer.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_class=target_class,
        features_to_vary=features_allowed_to_change
    )

    # Extract the counterfactual DataFrame
    counterfactuals_df = dice_result.cf_examples_list[0].final_cfs_df

    # Add the initial probability to the JSON
    dice_json = {
        "current_target_probability": f"{current_target_prob * 100:.1f}%"
    }

    for i in range(len(counterfactuals_df)):
        cf_row = counterfactuals_df.iloc[i]

        # Calculate the After probability for this specific counterfactual
        cf_features_only = cf_row.drop('target').to_frame().T

        # Ensure numeric features are floats
        for col in numeric_features:
            cf_features_only[col] = pd.to_numeric(cf_features_only[col])

        cf_probabilities = trained_model.predict_proba(cf_features_only)[0]
        cf_target_prob = cf_probabilities[target_class]

        #Calculate the Gain
        potential_gain = cf_target_prob - current_target_prob

        changes = []

        # Compare each feature with the original instance
        for feature in dataset_X.columns:
            original_value = query_instance[feature].iloc[0]
            new_value = cf_row[feature]

            # Skip if both values are NaN
            if pd.isna(original_value) and pd.isna(new_value):
                continue

            # Numeric features comparison (rounded for readability)
            if feature in numeric_features:
                original_val_rounded = round(float(original_value), 2)
                new_val_rounded = round(float(new_value), 2)

                if original_val_rounded != new_val_rounded:
                    changes.append({
                        "feature": feature,
                        "original_value": original_val_rounded,
                        "new_value": new_val_rounded
                    })

            # Categorical features comparison
            else:
                if str(original_value) != str(new_value):
                    changes.append({
                        "feature": feature,
                        "original_value": str(original_value),
                        "new_value": str(new_value)
                    })

        dice_json[f"option_{i+1}"] = {
            "new_target": target_label,
            "target_probability": f"{cf_target_prob * 100:.1f}%",
            "potential_gain": f"+{potential_gain * 100:.1f}%",
            "Changes": changes
        }

    dice_result.visualize_as_dataframe(show_only_changes=True)

    return dice_json

**Generate Prediction Report**

In [ ]:
def Save_Report(system_name, report):
    try:
        id = report["Prediction"]["record_id"]
        FBconn.put(f"/Reports/{system_name}", str(id), report)
        print(f"Report saved successfully in {system_name} for patient {id}.")
    except Exception as e:
        print(f"Firebase update error: {e}")

In [ ]:
def Generate_Prediction_Report(test_idx, X, Y, full_model, num_cols):
    # Select patient
    patient_row = X.iloc[[test_idx]]
    prediction = int(full_model.predict(patient_row)[0])

    patient_raw_data = [
        {
            "feature": str(col).replace("_", " ").title(),
            "value": str(X.iloc[test_idx][col])
        }
        for col in X.columns
    ]

    # Define categorical columns for DiCE
    categorical_columns = [col for col in X.columns if col not in num_cols]

    # Preprocess data once
    preprocessor_fitted = full_model.named_steps["preprocessor"]

    X_preprocessed = preprocessor_fitted.transform(X)
    feature_names = preprocessor_fitted.get_feature_names_out()
    features_labels = [name.split("__")[-1] for name in feature_names]

    patient_processed = preprocessor_fitted.transform(patient_row)

    # Run explainers
    lime_result = Get_Lime_Explantion(
        patient_processed,
        X_preprocessed,
        features_labels,
        full_model.named_steps["classifier"],
        record_id=test_idx
    )

    shap_result = Get_Shap_Explantion(
        patient_processed,
        X_preprocessed,
        features_labels,
        full_model.named_steps["classifier"],
        record_id=test_idx
    )

    dice_result = Get_Dice_Explantion(
        query_instance=patient_row,
        dataset_X=X,
        dataset_y=Y,
        trained_model=full_model,
        numeric_features=num_cols,
        categorical_features=categorical_columns
    )

    # Combine into final JSON
    report = {
        "Prediction": {
            "record_id": test_idx,
            "prediction": "Diabetic" if prediction == 1 else "Healthy",
            "description": "This AI-powered system analyzes patient medical records to predict if the patient is diabetic."},
        "Original_Data": patient_raw_data,
        "explanations": {
            "lime": lime_result,
            "shap": shap_result,
            "dice": dice_result
        }
    }

    return report

In [ ]:
report = Generate_Prediction_Report(
    test_idx=1,
    X=X,
    Y=Y,
    full_model=full_model,
    num_cols=num_cols
)
save_report = Save_Report("Diabities_System", report)

print(json.dumps(report, indent=2))